In [ ]:
# ============================================
# 0. SETUP & DRIVE
# ============================================
from google.colab import drive
drive.mount('/content/drive')

import os

# Clone QBERT
if not os.path.exists("QBERT"):
    !git clone https://github.com/sIncerass/QBERT.git

%cd QBERT

# Install dependencies
!pip install pytorch-pretrained-bert transformers datasets seqeval scipy scikit-learn -q


# ============================================
# 1. FIX QBERT LOADING ISSUE
# ============================================
!sed -i '/if len(error_msgs) > 0:/,+4c\        if len(error_msgs) > 0:\n            print("Skipping size mismatch")' modeling.py
!sed -i '/Skipping size mismatch/a\        return model' modeling.py
!sed -i 's/model.load_state_dict(state_dict)/model.load_state_dict(state_dict, strict=False)/g' modeling.py


# ============================================
# 2. DOWNLOAD QNLI DATA
# ============================================
if not os.path.exists("download_glue_data.py"):
    !wget https://gist.githubusercontent.com/W4ngatang/60c2bdb54d156a41194446737ce03e2e/raw/download_glue_data.py

!python download_glue_data.py --data_dir glue_data --tasks QNLI


# ============================================
# 3. QBERT TRAINING (QNLI)
# ============================================
!rm -rf train_output eval_output

print("\n Training QBERT on QNLI...\n")

!python run_classifier.py \
  --task_name qnli \
  --do_train \
  --do_lower_case \
  --data_dir glue_data/QNLI \
  --bert_model bert-base-uncased \
  --max_seq_length 128 \
  --train_batch_size 8 \
  --learning_rate 2e-5 \
  --num_train_epochs 3 \
  --output_dir train_output/


# ============================================
# 4. QBERT EVALUATION
# ============================================
!python run_classifier.py \
  --task_name qnli \
  --do_eval \
  --do_lower_case \
  --data_dir glue_data/QNLI \
  --bert_model train_output/ \
  --output_dir eval_output/

print("\n QBERT QNLI Results:")
!cat eval_output/eval_results.txt


# Save QBERT results
!cp -r train_output /content/drive/MyDrive/qbert_qnli_output
!cp -r eval_output /content/drive/MyDrive/qbert_qnli_eval_output


# ============================================
# 5. BERT BASELINE (QNLI)
# ============================================
print("\n Training BERT baseline on QNLI...\n")

from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
import numpy as np
from sklearn.metrics import accuracy_score

# Load dataset
dataset = load_dataset("glue", "qnli")

# Tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Tokenization (sentence pair)
def tokenize(example):
    return tokenizer(
        example["question"],
        example["sentence"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

dataset = dataset.map(tokenize, batched=True)

dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

# Model
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# Training args
training_args = TrainingArguments(
    output_dir="./bert_qnli",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    logging_steps=100
)

# Metric
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    compute_metrics=compute_metrics
)

trainer.train()

results = trainer.evaluate()
print("\n BERT QNLI Results:", results)

# Save BERT model
trainer.save_model("/content/drive/MyDrive/bert_qnli_model")

Streaming output truncated to the last 5000 lines.
Iteration:  62% 8158/13093 [42:46<26:45,  3.07it/s]
Iteration:  62% 8159/13093 [42:46<27:04,  3.04it/s]
Iteration:  62% 8160/13093 [42:47<27:11,  3.02it/s]
Iteration:  62% 8161/13093 [42:47<27:16,  3.01it/s]
Iteration:  62% 8162/13093 [42:47<27:35,  2.98it/s]
Iteration:  62% 8163/13093 [42:48<27:20,  3.00it/s]
Iteration:  62% 8164/13093 [42:48<26:34,  3.09it/s]
Iteration:  62% 8165/13093 [42:48<26:12,  3.13it/s]
Iteration:  62% 8166/13093 [42:48<25:49,  3.18it/s]
Iteration:  62% 8167/13093 [42:49<25:35,  3.21it/s]
Iteration:  62% 8168/13093 [42:49<25:43,  3.19it/s]
Iteration:  62% 8169/13093 [42:49<25:26,  3.23it/s]
Iteration:  62% 8170/13093 [42:50<25:15,  3.25it/s]
Iteration:  62% 8171/13093 [42:50<25:02,  3.28it/s]
Iteration:  62% 8172/13093 [42:50<24:59,  3.28it/s]
Iteration:  62% 8173/13093 [42:51<24:59,  3.28it/s]
Iteration:  62% 8174/13093 [42:51<24:51,  3.30it/s]
Iteration:  62% 8175/13093 [42:51<24:51,  3.30it/s]
Iteration:  6

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

qnli/train-00000-of-00001.parquet:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

qnli/validation-00000-of-00001.parquet:   0%|          | 0.00/872k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
!cat /content/drive/MyDrive/qbert_qnli_eval_output/eval_results.txt

acc = 0.9093904448105437
eval_loss = 0.3929998999503121
global_step = 0
loss = None


In [9]:
import torch
import time
import os
from transformers import BertForSequenceClassification

model_path = "/content/drive/MyDrive/qbert_qnli_output"

# Load model
model = BertForSequenceClassification.from_pretrained(model_path)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Dummy input
input_ids = torch.randint(0, 30522, (1, 128)).to(device)
attention_mask = torch.ones_like(input_ids).to(device)

# Warmup
for _ in range(10):
    _ = model(input_ids=input_ids, attention_mask=attention_mask)

# Measure latency
runs = 50
start = time.time()

for _ in range(runs):
    _ = model(input_ids=input_ids, attention_mask=attention_mask)

end = time.time()

latency = (end - start) / runs
print(f"\n Latency per sample: {latency:.6f} sec")


# -------- Model Size --------
total_size = 0
for dirpath, _, filenames in os.walk(model_path):
    for f in filenames:
        total_size += os.path.getsize(os.path.join(dirpath, f))

print(f" Model Size: {total_size / (1024*1024):.2f} MB")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/qbert_qnli_output
Key                                                      | Status     |  | 
---------------------------------------------------------+------------+--+-
bert.encoder.layer.{0...11}.attention.self.key.x_max     | UNEXPECTED |  | 
bert.encoder.layer.{0...11}.output.dense.x_max           | UNEXPECTED |  | 
bert.encoder.layer.{0...11}.attention.output.dense.x_min | UNEXPECTED |  | 
bert.encoder.layer.{0...11}.attention.self.query.x_min   | UNEXPECTED |  | 
bert.encoder.layer.{0...11}.attention.output.dense.x_max | UNEXPECTED |  | 
bert.encoder.layer.{0...11}.attention.self.value.x_min   | UNEXPECTED |  | 
bert.encoder.layer.{0...11}.output.dense.x_min           | UNEXPECTED |  | 
bert.encoder.layer.{0...11}.attention.self.query.x_max   | UNEXPECTED |  | 
bert.encoder.layer.{0...11}.attention.self.value.x_max   | UNEXPECTED |  | 
bert.encoder.layer.{0...11}.attention.self.key.x_min     | UNEXPECTED |  | 


 Latency per sample: 0.010364 sec
 Model Size: 418.65 MB
